# Smart MCQ Solver
**Approach:** TF-IDF Feature Engineering + Multi-Layer Neural Network (PyTorch)  
**Evaluation Metric:** MAP@3 (Mean Average Precision at 3)  


**Task:** Given a question prompt and 5 answer choices (A–E), predict the top-3 most likely correct answers in ranked order.

---
### Architecture Overview
```
Input: TF-IDF similarity features (5-dim) + TF-IDF cosine features (extended)
       ↓
 [Linear(5 → 256)] → ReLU → Dropout(0.3)
       ↓
 [Linear(256 → 128)] → ReLU → Dropout(0.3)
       ↓
 [Linear(128 → 64)] → ReLU
       ↓
 [Linear(64 → 5)]  → Softmax → Top-3 Predictions
```
The network is trained end-to-end on TF-IDF derived features, with cross-entropy loss.
Experiment tracking is done via Weights & Biases (W&B).

## 1. Install & Import Dependencies

In [4]:
import wandb

wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/spreadsheets600/.netrc


True

In [8]:
import os
import string

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from tqdm import tqdm


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device : {DEVICE}")
print(
    f"Device Name : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}"
)
print(f"\nPyTorch Version : {torch.__version__}")

Using Device : cuda
Device Name : NVIDIA GeForce RTX 2050

PyTorch Version : 2.13.0+cu130


## 2. Data Loading & Exploration

In [12]:
# Kaggle Paths
# TRAIN_PATH = "/kaggle/input/smart-mcq-solver-challenge/train.csv"
# TEST_PATH  = "/kaggle/input/smart-mcq-solver-challenge/test.csv"

# Local Testing Paths

TRAIN_PATH = "/home/spreadsheets600/Projects/DL-GenAI-Project/Data/train.csv"
TEST_PATH = "/home/spreadsheets600/Projects/DL-GenAI-Project/Data/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=" * 55)
print("Training Set")
print("=" * 55)
print(train_df.info())
print("=" * 55)
print(train_df.head(3))

Training Set
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2000 non-null   int64
 1   prompt  2000 non-null   str  
 2   A       2000 non-null   str  
 3   B       2000 non-null   str  
 4   C       2000 non-null   str  
 5   D       2000 non-null   str  
 6   E       2000 non-null   str  
 7   answer  2000 non-null   str  
dtypes: int64(1), str(7)
memory usage: 1.9 MB
None
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   

                         

In [13]:
# Basic EDA
print("Answer Distribution ( Training Set ) :")
print(train_df["answer"].value_counts())
print()

print(f"Train Size : {len(train_df):,}")
print(f"Test Size : {len(test_df):,}")

print(f"Missing values :\n{train_df.isnull().sum()}")

Answer Distribution ( Training Set ) :
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Train Size : 2,000
Test Size : 500
Missing values :
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64


## 3. Text Preprocessing

In [15]:
def clean_text(text: str):
    """
    Basic Text Cleaning Pipeline :

    - Lowercase
    - Remove Punctuation
    - Strip Extra Whitespace
    """

    if not isinstance(text, str):
        return ""

    translator = str.maketrans("", "", string.punctuation)
    text = text.lower().translate(translator)
    text = " ".join(text.split())

    return text


OPTIONS = ["A", "B", "C", "D", "E"]


def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply clean_text To The Prompt And Options Columns.
    Returns A Copy With New Columns Added For Cleaned Text.
    """

    df = df.copy()
    df["prompt_clean"] = df["prompt"].apply(clean_text)

    for opt in OPTIONS:
        df[f"{opt}_clean"] = df[opt].apply(clean_text)

    return df


train_df = preprocess_df(train_df)
test_df = preprocess_df(test_df)

# Verify Cleaning
print("Sample Cleaned Prompt :")
print(train_df["prompt_clean"].iloc[0][:120])

Sample Cleaned Prompt :
pick the best possible answer what is martin heideggers view on the relationship between time and human existence among 


## 4. TF-IDF Vectorizer

In [17]:
corpus_parts = [train_df["prompt_clean"]]

for opt in OPTIONS:
    corpus_parts.append(train_df[f"{opt}_clean"])

fit_corpus = pd.concat(corpus_parts, ignore_index=True)
tfidf = TfidfVectorizer(
    stop_words="english",  # Remove English Stop Words
    ngram_range=(1, 2),  # Unigrams + Bigrams  
    max_features=8000,  # Cap Vocabulary Size
    sublinear_tf=True,  # Apply log(1 + tf)
)
tfidf.fit(fit_corpus)

print(f"TF-IDF Vocabulary Size : {len(tfidf.vocabulary_):,}")
print(f"Corpus Used For Fitting : {len(fit_corpus):,} Documents")

TF-IDF Vocabulary Size : 8,000
Corpus Used For Fitting : 12,000 Documents


## 5. Feature Engineering

In [ ]:
def build_features(df: pd.DataFrame, vectorizer: TfidfVectorizer):
    """
    Build A (N, 5) Feature Matrix Where Each Row Is:
        [cos_sim(prompt, A), cos_sim(prompt, B), ..., cos_sim(prompt, E)]

    Parameters
    ----------
    df          : DataFrame Containing Cleaned Prompt And Option Columns.
    vectorizer  : Fitted TfidfVectorizer.

    Returns
    -------
    features    : np.ndarray Of Shape (N, 5)
    """

    # Transform All Prompts At Once (sparse matrix, N × vocab)
    prompt_vecs = vectorizer.transform(df["prompt_clean"])

    feature_cols = []
    for opt in OPTIONS:
        option_vecs = vectorizer.transform(df[f"{opt}_clean"])

        # Row-Wise Cosine Similarity: shape (N,)
        sims = np.array(
            [
                cosine_similarity(prompt_vecs[i], option_vecs[i])[0, 0]
                for i in range(len(df))
            ]
        )
        feature_cols.append(sims)

    # Stack Into (N, 5)
    return np.column_stack(feature_cols)


print("Building Training Features...")
X_train_full = build_features(train_df, tfidf)

print(f"Training Feature Matrix Shape : {X_train_full.shape}")

print("Building Test Features...")
X_test = build_features(test_df, tfidf)

print(f"Test Feature Matrix Shape : {X_test.shape}")

Building Training Features...
Training Feature Matrix Shape : (2000, 5)
Building Test Features...
Test Feature Matrix Shape : (500, 5)


In [20]:
label_encoder = LabelEncoder()
label_encoder.fit(["A", "B", "C", "D", "E"])

y_train_full = label_encoder.transform(train_df["answer"])
print(f"Label Classes : {label_encoder.classes_}")
print(f"y_train Shape : {y_train_full.shape}")

Label Classes : ['A' 'B' 'C' 'D' 'E']
y_train Shape : (2000,)


In [21]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_full   # Preserver Class Distribution In Train/Val Splits
)

print(f"Train Split Size : {len(X_tr):,}")
print(f"Validation  Split Size : {len(X_val):,}")

Train Split Size : 1,700
Validation  Split Size : 300


## 6. MAP@3 Metric

In [30]:
def map_at_k(actual: list, predicted: list, k: int = 3):
    """
    Compute Mean Average Precision At K (MAP@K).

    Parameters
    ----------
    actual    : List Of True Answer Labels (Strings Or Ints).
    predicted : List Of Lists - Each Inner List Is The Ranked Top-K Predictions.
    k         : Rank Cutoff.

    Returns
    -------
    float : MAP@K Score In [0, 1].
    """

    scores = []
    for act, preds in zip(actual, predicted):
        ap = 0.0
        for rank, pred in enumerate(preds[:k], start=1):
            if pred == act:
                ap = 1.0 / rank

                break  # Single-Answer Task

        scores.append(ap)
    return float(np.mean(scores))


# Quick Sanity Check
assert abs(map_at_k(["A"], [["A", "B", "C"]], k=3) - 1.0) < 1e-9
assert abs(map_at_k(["A"], [["B", "A", "C"]], k=3) - 0.5) < 1e-9
assert abs(map_at_k(["A"], [["C", "D", "A"]], k=3) - 1 / 3) < 1e-9
assert abs(map_at_k(["A"], [["B", "C", "D"]], k=3) - 0.0) < 1e-9

print("MAP@3 Function Passed All Checks")

MAP@3 Function Passed All Checks


## 7. Neural Network Architecture

In [23]:
class MCQNeuralNet(nn.Module):
    """
    Feed-Forward Neural Network For MCQ Answer Scoring.

    Input  : 5-Dimensional TF-IDF Cosine Similarity Features.
    Output : 5-Dimensional Logits.
    """

    def __init__(self, input_dim: int = 5, dropout_rate: float = 0.3):
        super(MCQNeuralNet, self).__init__()

        self.network = nn.Sequential(
            # Layer 1
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            # Layer 2
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            # Layer 3
            nn.Linear(128, 64),
            nn.ReLU(),
            # Output Layer
            nn.Linear(64, 5),
        )

    def forward(self, x: torch.Tensor):
        """
        Forward Pass.

        Parameters
        ----------
        x : Tensor Of Shape (batch_size, input_dim)

        Returns
        -------
        Tensor Of Shape (batch_size, 5) — raw logits.
        """

        return self.network(x)


model_nn = MCQNeuralNet(input_dim=5, dropout_rate=0.3).to(DEVICE)
print(model_nn)
print()

total_params = sum(p.numel() for p in model_nn.parameters() if p.requires_grad)
print(f"Total Trainable Parameters : {total_params:,}")

MCQNeuralNet(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): ReLU()
    (10): Linear(in_features=64, out_features=5, bias=True)
  )
)

Total Trainable Parameters : 43,781


## 8. Prepare PyTorch DataLoaders

In [24]:
def make_dataloader(
    X: np.ndarray, y: np.ndarray | None, batch_size: int = 64, shuffle: bool = True
):
    X_tensor = torch.tensor(X, dtype=torch.float32)

    if y is not None:
        y_tensor = torch.tensor(y, dtype=torch.long)
        dataset = TensorDataset(X_tensor, y_tensor)

    else:
        dataset = TensorDataset(X_tensor)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


BATCH_SIZE = 64

train_loader = make_dataloader(X_tr, y_tr, batch_size=BATCH_SIZE, shuffle=True)
val_loader = make_dataloader(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)
test_loader = make_dataloader(X_test, None, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train Batches : {len(train_loader)}")
print(f"Validation Batches : {len(val_loader)}")
print(f"Test Batches : {len(test_loader)}")

Train Batches : 27
Validation Batches : 5
Test Batches : 8


## 9. Training Loop with W&B Tracking

In [ ]:
HYPERPARAMS = {
    "model": "TF-IDF + Neural Network",
    "epochs": 60,
    "batch_size": BATCH_SIZE,
    "learning_rate": 3e-3,
    "weight_decay": 1e-4,
    "dropout_rate": 0.3,
    "optimizer": "Adam",
    "scheduler": "CosineAnnealingLR",
    "tfidf_ngrams": "(1,2)",
    "tfidf_max_features": 8000,
    "seed": SEED,
}

run = wandb.init(
    config=HYPERPARAMS,
    project="24F2008474-t22026",
    name="tfidf-nn-scratch-run-1",
    tags=["from-scratch", "tfidf", "neural-network"],
)

print("W&B Run Initialised :", run.name)

W&B Run Initialised : tfidf-nn-scratch-run-1


wandb: WARNING Artifact "submission-tfidf-nn" already exists with the same content. No new version will be created.


In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model_nn.parameters(),
    lr=HYPERPARAMS["learning_rate"],
    weight_decay=HYPERPARAMS["weight_decay"],
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HYPERPARAMS["epochs"])

print("Optimiser : Adam")
print("Criterion : CrossEntropyLoss")
print("Scheduler : CosineAnnealingLR")

Optimiser : Adam
Criterion : CrossEntropyLoss
Scheduler : CosineAnnealingLR


In [31]:
def get_top3_preds(logits: np.ndarray):
    ranked_indices = np.argsort(logits, axis=1)[:, ::-1]
    return [[OPTIONS[idx] for idx in row[:3]] for row in ranked_indices]


def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()

    all_logits = []
    all_labels = []
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            X_batch, y_batch = batch[0].to(DEVICE), batch[1].to(DEVICE)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item()
            n_batches += 1

            all_logits.append(logits.cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())

    all_logits = np.vstack(all_logits)  # (N, 5)
    all_labels = np.concatenate(all_labels)  # (N,)

    # Accuracy
    preds_top1 = np.argmax(all_logits, axis=1)
    accuracy = float(np.mean(preds_top1 == all_labels))

    # MAP@3
    preds_top3 = get_top3_preds(all_logits)
    actual_labels = [OPTIONS[i] for i in all_labels]
    map3 = map_at_k(actual_labels, preds_top3, k=3)

    return {
        "accuracy": accuracy,
        "map3": map3,
        "loss": total_loss / n_batches,
    }

In [32]:
EPOCHS = HYPERPARAMS["epochs"]
best_map3 = 0.0
best_model_state = None

print(f"Starting Training For {EPOCHS} Epochs On {DEVICE}\n")
print(
    f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val MAP@3':>9}"
)
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    model_nn.train()
    train_loss = 0.0
    n_batches = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model_nn(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        n_batches += 1

    scheduler.step()  # Updating Learning Rate

    avg_train_loss = train_loss / n_batches
    current_lr = scheduler.get_last_lr()[0]

    val_metrics = evaluate(model_nn, val_loader)

    wandb.log(
        {
            "epoch": epoch,
            "train/loss": avg_train_loss,
            "val/loss": val_metrics["loss"],
            "val/accuracy": val_metrics["accuracy"],
            "val/map3": val_metrics["map3"],
            "learning_rate": current_lr,
        }
    )

    if val_metrics["map3"] > best_map3:
        best_map3 = val_metrics["map3"]
        best_model_state = {k: v.clone() for k, v in model_nn.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"{epoch:>6} | {avg_train_loss:>10.4f} | "
            f"{val_metrics['loss']:>8.4f} | "
            f"{val_metrics['accuracy']:>7.4f} | "
            f"{val_metrics['map3']:>9.4f}"
        )

print("-" * 55)
print(f"\nBest Validation MAP@3 : {best_map3:.4f}")

Starting Training For 60 Epochs On cuda

 Epoch | Train Loss | Val Loss | Val Acc | Val MAP@3
-------------------------------------------------------
     1 |     1.4825 |   1.4758 |  0.3467 |    0.5222
    10 |     1.2005 |   1.1005 |  0.5567 |    0.7011
    20 |     1.0861 |   1.0017 |  0.6033 |    0.7217
    30 |     0.9631 |   0.8391 |  0.7033 |    0.7894
    40 |     0.9086 |   0.7603 |  0.7133 |    0.8033
    50 |     0.9073 |   0.7325 |  0.7400 |    0.8178
    60 |     0.8884 |   0.7340 |  0.7367 |    0.8161
-------------------------------------------------------

Best Validation MAP@3 : 0.8250


## 10. Final Evaluation on Validation Set

In [33]:
model_nn.load_state_dict(best_model_state)
model_nn.eval()

final_val_metrics = evaluate(model_nn, val_loader)

print("=" * 55)
print(" FINAL VALIDATION METRICS")
print("=" * 55)

print(f"  MAP@3 : {final_val_metrics['map3']:.4f}")
print(f"  Accuracy : {final_val_metrics['accuracy']:.4f}")
print(f"  Val Loss : {final_val_metrics['loss']:.4f}")

wandb.summary["best_val_map3"] = final_val_metrics["map3"]
wandb.summary["best_val_accuracy"] = final_val_metrics["accuracy"]

 FINAL VALIDATION METRICS
  MAP@3 : 0.8250
  Accuracy : 0.7467
  Val Loss : 0.7342


## 11. Error Analysis

In [34]:
model_nn.eval()
val_logits_list = []
val_labels_list = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        logits = model_nn(X_batch.to(DEVICE)).cpu().numpy()
        val_logits_list.append(logits)
        val_labels_list.append(y_batch.numpy())

val_logits = np.vstack(val_logits_list)
val_labels = np.concatenate(val_labels_list)

preds_top1 = np.argmax(val_logits, axis=1)
preds_top3 = get_top3_preds(val_logits)

_, val_idx = train_test_split(
    range(len(train_df)), test_size=0.15, random_state=SEED, stratify=y_train_full
)
val_df = train_df.iloc[val_idx].reset_index(drop=True)

error_df = val_df[["prompt", "answer"]].copy()

error_df["true_label"] = [OPTIONS[i] for i in val_labels]
error_df["top1_pred"] = [OPTIONS[i] for i in preds_top1]
error_df["top3_preds"] = [",".join(p) for p in preds_top3]
error_df["top1_correct"] = error_df["true_label"] == error_df["top1_pred"]

error_df["in_top3"] = [
    true in p3 for true, p3 in zip(error_df["true_label"], preds_top3)
]

print(f"Top-1 Accuracy : {error_df['top1_correct'].mean():.4f}")
print(f"Top-3 Accuracy : {error_df['in_top3'].mean():.4f}")

print()

print("Sample Misclassified Questions :")
misclassified = error_df[~error_df["top1_correct"]]

print(
    misclassified[["prompt", "true_label", "top1_pred", "top3_preds"]]
    .head(5)
    .to_string(index=False)
)

Top-1 Accuracy : 0.7467
Top-3 Accuracy : 0.9267

Sample Misclassified Questions :
                                                                                                                                                                           prompt true_label top1_pred top3_preds
                                                                               Pick the best possible answer: What is the Minkowski diagram used for? based on the given context.          E         B      B,E,D
Select the most accurate option: What are some models that attempt to account for all observations without invoking supplemental non-baryonic matter? from the following choices.          C         B      B,C,A
                                        Determine the correct option: What is the main focus of the Environmental Science Center at Qatar University? from the following choices.          D         B      B,C,D
                                                                  Determine th

## 12. Generate Test Predictions & Kaggle Submission

In [35]:
model_nn.eval()
test_logits_list = []

with torch.no_grad():
    for batch in test_loader:
        X_batch = batch[0].to(DEVICE)
        logits = model_nn(X_batch).cpu().numpy()
        test_logits_list.append(logits)

test_logits = np.vstack(test_logits_list)  # (N_test, 5)
test_preds = get_top3_preds(test_logits)  # list of [top1, top2, top3]


submission_df = pd.DataFrame(
    {"id": test_df["id"], "prediction": [" ".join(p) for p in test_preds]}
)

print(f"Submission Shape : {submission_df.shape}")
print(submission_df.head(10).to_string(index=False))

Submission Shape : (500, 2)
 id prediction
  1      B C A
  2      B C A
  3      B C A
  4      E B D
  5      B C D
  6      D B A
  7      E B A
  8      B C A
  9      B C D
 10      B D C


In [ ]:
# Kaggle
# SUBMISSION_PATH = "submission.csv"

# Local Testing
SUBMISSION_PATH = "Submissions/submission_tfidf_nn.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission Saved To : {SUBMISSION_PATH}")

artifact = wandb.Artifact("submission-tfidf-nn", type="submission")
artifact.add_file(SUBMISSION_PATH)
wandb.log_artifact(artifact)

print("Submission Artifact Loggged To W&B")

Submission Saved To : submission_tfidf_nn.csv
Submission Artifact Loggged To W&B


## 13. Save Model Weights & Finish W&B Run

In [ ]:
# Kaggle
# MODEL_PATH = "tfidf_nn_model.pt"

# Lcoal Testing
MODEL_PATH = "Models/tfidf_nn_model.pt"

torch.save(
    {
        "model_state_dict": model_nn.state_dict(),
        "hyperparams": HYPERPARAMS,
        "best_val_map3": best_map3,
    },
    MODEL_PATH,
)
print(f"Model Checkpoint Saved : {MODEL_PATH}")


model_artifact = wandb.Artifact("tfidf-nn-model", type="model")
model_artifact.add_file(MODEL_PATH)

wandb.init()
wandb.log_artifact(model_artifact)

wandb.finish()
print("\nW&B Run Finished")

Model Checkpoint Saved : tfidf_nn_model.pt


wandb: ERROR The nbformat package was not found. It is required to save notebook history.



W&B Run Finished
